# Stage 2B — Waveform and effective-operator forensics

Frozen evaluation only. This notebook loads the exact Stage-1A.2 checkpoint, performs no optimization, and uses the unchanged DD-v1 and effective-operator conventions. Matrix images use row 0 at the top without transforming the underlying matrix.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import Image, display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.stage2b import OUT, run_stage2b

In [ ]:
diagnostics = run_stage2b()
diagnostics['checkpoint'], diagnostics['afdm']['integer_path_gate_passed']

## Orientation and representative physical channel

In [ ]:
display(Image(filename=str(OUT / 'fig00_orientation_sanity.png')))
display(Image(filename=str(OUT / 'fig03_corrected_representative_operator.png')))
display(Image(filename=str(OUT / 'fig04_representative_path_resolved.png')))

A positive circular shift appears one pixel below the main diagonal with wrap at the top-right. No transpose, reversal, or conjugation is applied. On the median-control DD-v1 channel, OFDM and AFDM remain diagonal-band representations, OTFS produces repeated shifted lines, and Learned confines each physical path to the same three dense subspaces seen in the aggregate operator. Path colors overlap inside those subspaces, so the blocks are not single-path supports.

## Actual transmitted atoms and dense spectra

In [ ]:
display(Image(filename=str(OUT / 'fig01_representative_time_atoms.png')))
display(Image(filename=str(OUT / 'fig02_dense_atom_spectra.png')))
display(pd.read_csv(OUT / 'representative_atom_selection.csv'))

OFDM atoms are constant-envelope windowed sinusoids and recover translated Dirichlet/sinc-like lobes. AFDM atoms are constant-envelope chirps with broadband dense spectra. OTFS atoms are periodic sparse pulse trains with repeated spectral lobes. Learned basis atoms have nonconstant, irregular envelopes and moderately localized, often multi-lobed bandpass spectra. The Learned atoms within a band are not established as translated copies of one prototype.

## Controlled single paths and AFDM audit

In [ ]:
display(Image(filename=str(OUT / 'fig05_single_path_gallery.png')))
display(pd.read_csv(OUT / 'afdm_integer_path_audit.csv'))
display(pd.read_csv(OUT / 'afdm_progression_path_centers.csv'))

With the project-tuned AFDM parameters, every integer path lands exactly at column-minus-row offset `(delay + Doppler) mod 64`. Fractional Doppler broadens a one-bin diagonal to 3–7 bins for 95% offset energy; fractional delay increases the observed widths to 7–13 bins. Learned instead maps even a single nonidentity path into dense within-group blocks. These images verify implementation behavior, not superiority.

## Integer to fractional-Doppler to fully off-grid progression

In [ ]:
for name in ('fig06a_progression_grid_dd.png', 'fig06b_progression_fractional_doppler.png', 'fig06c_progression_full_offgrid.png'):
    display(Image(filename=str(OUT / name)))

Grid alignment gives exact shifted structures for the classical bases. Fractional Doppler creates leakage around those centers; fractional delay adds further spreading. Learned's three subspaces persist through all regimes, although within-block textures and cross-block leakage change. Thus the blocks belong to the frozen basis, not to one favorable DD-v1 realization.

## Reference-style four-path illustration

In [ ]:
display(Image(filename=str(OUT / 'fig07a_reference_style_grid_four_path.png')))
display(Image(filename=str(OUT / 'fig07b_reference_style_fractional_doppler_four_path.png')))
display(pd.read_csv(OUT / 'reference_style_four_path.csv'))

This is a pedagogical reference-style channel, not an exact paper reproduction. Distinct integer paths are cleanly separated under AFDM's measured support law. Replacing only Dopplers by fractional values visibly broadens those colored diagonals while preserving their centers.

## Frozen DD-v1 easy / median / hard channels

In [ ]:
display(Image(filename=str(OUT / 'fig08_ddv1_easy_median_hard.png')))
display(pd.read_csv(OUT / 'ddv1_easy_median_hard_channels.csv'))

Channels are selected only from IDENTITY K=24 difficulty at the 10th, 50th, and 90th percentiles. Learned retains the same three block locations at every difficulty; harder channels change intensity and leakage, not the partition. This rules out channel 189 as the source of the block layout.

## Full-bank energy, top-24 support, and spectral association

In [ ]:
display(Image(filename=str(OUT / 'fig09_ensemble_average_operator_energy.png')))
display(Image(filename=str(OUT / 'fig10_top24_support_probability.png')))
display(Image(filename=str(OUT / 'fig11_learned_spectral_operator_association.png')))
community = json.load((OUT / 'learned_community_summary.json').open())
pd.DataFrame([{k: v for k, v in community.items() if not isinstance(v, (list, dict))}])

The three Learned blocks survive averaging over all 1,024 held-out channels and dominate the top-24 support map. Unsupervised operator coupling selects groups of 20, 21, and 23 atoms; 96.46% of ensemble energy and 83.88% of expected top-24 support lie within those groups. Spectral overlap and operator coupling have rank correlation about 0.950 and independently clustered labels have adjusted Rand index about 0.951. Centroid sorting makes the circular frequency grouping cleaner, with a necessary split at the ±0.5 wrap. These associations support local spectral subspaces compatible in size with K=24, but do not establish that K=24 caused them.

## What remains open

The data support persistent, spectrally associated local subspaces. They do not yet distinguish whether the partition is caused by the receiver budget, the three-path DD-v1 distribution, initialization/optimization symmetry breaking, or a more general finite-dimensional filter-bank solution. No waveform-family formula is fit in this stage.